# Pipeline Landing para Bronze — CineData Analytics

Este notebook realiza a ingestão dos dados brutos da camada **Landing** (arquivos CSV e API do Banco Central) para a camada **Bronze** em formato Parquet.

### Padrões Técnicos Aplicados:
- **Esquemas Canônicos e Validação**: Definição de `StructType` com validação de conformidade de colunas para cada fonte de dados.
- **Constantes de Colunas com Acesso Estático**: Uso de `@dataclass(frozen=True)` acessadas diretamente na classe.
- **Rastreabilidade**: Adição do carimbo de data e hora (`ingestion_datetime`) em todas as tabelas persistidas na Bronze.

In [ ]:
import os
import sys
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import requests
from pydantic import ConfigDict, Field
from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from sparkdantic import SparkModel

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

DATA_DIR = Path.cwd() / "data"
LANDING_DIR = DATA_DIR / "landing"
BRONZE_DIR = DATA_DIR / "bronze"

BRONZE_DIR.mkdir(exist_ok=True)

spark = (
    SparkSession
    .builder
    .appName("Landing_to_Bronze")
    .getOrCreate()
)

def enforce_dataframe_schema(
    dataframe: DataFrame,
    expected_schema: StructType,
    strict_columns: bool = True
) -> DataFrame:
    """
    Valida a presença de todos os campos definidos no StructType e reordena as colunas.
    Em modo estrito, impede a persistência de colunas imprevistas no contrato.
    """
    actual_column_names = set(dataframe.columns)
    expected_column_names = [field.name for field in expected_schema.fields]
    missing_columns = set(expected_column_names) - actual_column_names

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {missing_columns}")

    if strict_columns:
        unexpected_columns = actual_column_names - set(expected_column_names)
        if unexpected_columns:
            raise ValueError(f"Colunas imprevistas encontradas no DataFrame: {unexpected_columns}")

    ordered_column_expressions = [
        col(field.name).cast(field.dataType).alias(field.name)
        for field in expected_schema.fields
    ]
    return dataframe.select(ordered_column_expressions)

def write_dataframe_with_timestamp(
    dataframe: DataFrame,
    target_path: Path,
    storage_format: str = "parquet",
    save_mode: str = "overwrite"
) -> None:
    """
    Persiste o DataFrame com a coluna ingestion_datetime carimbando o momento de gravação.
    """
    dataframe_with_timestamp = dataframe.withColumn("ingestion_datetime", current_timestamp())
    (
        dataframe_with_timestamp.write
        .format(storage_format)
        .mode(save_mode)
        .save(str(target_path))
    )


## 1. Ingestão de Arquivos CSV da Camada Landing com Validação de Esquema

Cada conjunto de dados da Landing possui uma `dataclass(frozen=True)` com os nomes das colunas e um `StructType` canônico que valida a conformidade dos dados antes da escrita na camada Bronze.

In [ ]:
@dataclass(frozen=True)
class RawMoviesInfoColumns:
    ID: str = "id"
    TCONST: str = "tconst"
    TITLE: str = "title"
    ORIGINAL_TITLE: str = "original_title"
    ORIGINAL_LANGUAGE: str = "original_language"
    RELEASE_DATE: str = "release_date"
    RUNTIME: str = "runtime"
    STATUS: str = "status"
    OVERVIEW: str = "overview"
    TAGLINE: str = "tagline"

RawMoviesInfoBronzeSchema = StructType([
    StructField(RawMoviesInfoColumns.ID, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.TCONST, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.TITLE, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.ORIGINAL_TITLE, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.ORIGINAL_LANGUAGE, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.RELEASE_DATE, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.RUNTIME, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.STATUS, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.OVERVIEW, StringType(), nullable=True),
    StructField(RawMoviesInfoColumns.TAGLINE, StringType(), nullable=True),
])

@dataclass(frozen=True)
class RawMoviesFinancialsColumns:
    ID: str = "id"
    BUDGET: str = "budget"
    REVENUE: str = "revenue"

RawMoviesFinancialsBronzeSchema = StructType([
    StructField(RawMoviesFinancialsColumns.ID, StringType(), nullable=True),
    StructField(RawMoviesFinancialsColumns.BUDGET, StringType(), nullable=True),
    StructField(RawMoviesFinancialsColumns.REVENUE, StringType(), nullable=True),
])

@dataclass(frozen=True)
class RawMoviesMetricsColumns:
    ID: str = "id"
    POPULARITY: str = "popularity"
    VOTE_AVERAGE: str = "vote_average"
    VOTE_COUNT: str = "vote_count"
    AVERAGE_RATING: str = "averageRating"
    NUM_VOTES: str = "numVotes"

RawMoviesMetricsBronzeSchema = StructType([
    StructField(RawMoviesMetricsColumns.ID, StringType(), nullable=True),
    StructField(RawMoviesMetricsColumns.POPULARITY, StringType(), nullable=True),
    StructField(RawMoviesMetricsColumns.VOTE_AVERAGE, StringType(), nullable=True),
    StructField(RawMoviesMetricsColumns.VOTE_COUNT, StringType(), nullable=True),
    StructField(RawMoviesMetricsColumns.AVERAGE_RATING, StringType(), nullable=True),
    StructField(RawMoviesMetricsColumns.NUM_VOTES, StringType(), nullable=True),
])

@dataclass(frozen=True)
class RawCreditsAndTagsColumns:
    ID: str = "id"
    GENRES: str = "genres"
    PRODUCTION_COMPANIES: str = "production_companies"
    PRODUCTION_COUNTRIES: str = "production_countries"
    SPOKEN_LANGUAGES: str = "spoken_languages"
    KEYWORDS: str = "keywords"
    DIRECTORS: str = "directors"
    WRITERS: str = "writers"
    CAST: str = "cast"

RawCreditsAndTagsBronzeSchema = StructType([
    StructField(RawCreditsAndTagsColumns.ID, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.GENRES, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.PRODUCTION_COMPANIES, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.PRODUCTION_COUNTRIES, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.SPOKEN_LANGUAGES, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.KEYWORDS, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.DIRECTORS, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.WRITERS, StringType(), nullable=True),
    StructField(RawCreditsAndTagsColumns.CAST, StringType(), nullable=True),
])

@dataclass(frozen=True)
class RawMoviesReviewsColumns:
    ID: str = "id"
    NOME: str = "nome"
    NOTA: str = "nota"
    COMENTARIO: str = "comentario"

RawMoviesReviewsBronzeSchema = StructType([
    StructField(RawMoviesReviewsColumns.ID, StringType(), nullable=True),
    StructField(RawMoviesReviewsColumns.NOME, StringType(), nullable=True),
    StructField(RawMoviesReviewsColumns.NOTA, StringType(), nullable=True),
    StructField(RawMoviesReviewsColumns.COMENTARIO, StringType(), nullable=True),
])

landing_ingestion_configurations = [
    ("movies_info_TMDB_IMDB.csv", "bronze.tb_movies_info", RawMoviesInfoBronzeSchema),
    ("movies_financials_IMDB_TMDB.csv", "bronze.tb_movies_financials", RawMoviesFinancialsBronzeSchema),
    ("movies_metrics_IMDB_TMDB.csv", "bronze.tb_movies_metrics", RawMoviesMetricsBronzeSchema),
    ("credits_and_tags_IMDB_TMDB.csv", "bronze.tb_credits_and_tags", RawCreditsAndTagsBronzeSchema),
    ("movies_reviews.csv", "bronze.tb_movies_reviews", RawMoviesReviewsBronzeSchema),
]

for landing_filename, bronze_table_name, target_schema in landing_ingestion_configurations:
    raw_csv_dataframe = spark.read.csv(
        str(LANDING_DIR / landing_filename),
        header=True,
        inferSchema=False
    )
    validated_dataframe = enforce_dataframe_schema(raw_csv_dataframe, target_schema)
    write_dataframe_with_timestamp(validated_dataframe, BRONZE_DIR / bronze_table_name)

## 2. Ingestão da API do Banco Central (Cotação do Dólar)

- **Formato de Data**: `MM-DD-AAAA` conforme contrato da API Olinda.
- **Validação de Modelo**: Conversão e validação estrita com Pydantic e `StructType` canônico para gravação em `bronze.tb_cotacao_dolar`.

In [ ]:
@dataclass(frozen=True)
class BacenCotacaoColumns:
    COTACAO_COMPRA: str = "cotacaoCompra"
    DATA_HORA_COTACAO: str = "dataHoraCotacao"

BacenCotacaoBronzeSchema = StructType([
    StructField(BacenCotacaoColumns.COTACAO_COMPRA, DoubleType(), nullable=True),
    StructField(BacenCotacaoColumns.DATA_HORA_COTACAO, StringType(), nullable=True),
])

class CotacaoItem(SparkModel):
    model_config = ConfigDict(populate_by_name=True)
    
    cotacao_compra: float = Field(alias="cotacaoCompra")
    data_hora_cotacao: datetime = Field(alias="dataHoraCotacao")

class PtaxResponse(SparkModel):
    model_config = ConfigDict(populate_by_name=True)

    odata_context: str = Field(alias="@odata.context")
    value: list[CotacaoItem]

In [ ]:
BACEN_API_DATE_FORMAT = "%m-%d-%Y"
QUERY_INTERVAL_DAYS = 7
RECIFE_TIMEZONE = "America/Recife"

DOLAR_QUOTE_ENDPOINT = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    "@dataInicial='{start_date}'&@dataFinalCotacao='{end_date}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

current_datetime_recife = datetime.now(ZoneInfo(RECIFE_TIMEZONE))
start_period_date = current_datetime_recife - timedelta(days=QUERY_INTERVAL_DAYS)

formatted_end_date = current_datetime_recife.strftime(BACEN_API_DATE_FORMAT)
formatted_start_date = start_period_date.strftime(BACEN_API_DATE_FORMAT)

response = requests.get(
    DOLAR_QUOTE_ENDPOINT.format(
        start_date=formatted_start_date, end_date=formatted_end_date
    )
)
response.raise_for_status()
validated_api_response = PtaxResponse.model_validate(response.json())

serialized_quote_records = [
    {
        BacenCotacaoColumns.COTACAO_COMPRA: item.cotacao_compra,
        BacenCotacaoColumns.DATA_HORA_COTACAO: item.data_hora_cotacao.strftime("%Y-%m-%d %H:%M:%S"),
    }
    for item in validated_api_response.value
]

dataframe_cotacao_raw = spark.createDataFrame(data=serialized_quote_records, schema=BacenCotacaoBronzeSchema)
dataframe_cotacao_validated = enforce_dataframe_schema(dataframe_cotacao_raw, BacenCotacaoBronzeSchema)
write_dataframe_with_timestamp(dataframe_cotacao_validated, BRONZE_DIR / "bronze.tb_cotacao_dolar")
dataframe_cotacao_validated.show(truncate=False)